In [3]:
# Import libraries dasar
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Tentukan path (sesuaikan dengan struktur folder Anda)
# '..' berarti "naik satu folder"
DATA_PATH = r'C:\FRAUD-Sinta2\Data\Olist'
OUTPUT_PATH = r'C:\FRAUD-Sinta2\Output'

print("Libraries imported.")

Libraries imported.


In [5]:
try:
    df_orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
    df_payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
    print("Dataset Olist berhasil di-load")
except FileNotFoundError:
    print(f"Error: File CSV ada di Folder : {DATA_PATH}")

# Tampilkan 5 baris pertama untuk diperiksa
print("Data Orders:")
print(df_orders.head())
print("\nData Payments:")
print(df_payments.head())

# Ambil kolom yang relevan saja
df_orders = df_orders[['order_id', 'customer_id', 'order_purchase_timestamp']]
df_payments = df_payments[['order_id', 'payment_value']]

# Gabungkan (merge) kedua tabel menggunakan 'order_id'
df_merged = pd.merge(df_orders, df_payments, on='order_id', how='inner')

print(f"\nData berhasil digabung. Total transaksi: {len(df_merged)}")

Dataset Olist berhasil di-load
Data Orders:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1   

In [6]:
# Ganti nama kolom agar mudah dibaca (sesuai kerangka)
df_merged.rename(columns={
    'customer_id': 'user_id',
    'order_purchase_timestamp': 'timestamp',
    'payment_value': 'amount'
}, inplace=True)

# 1. Konversi timestamp (string) menjadi objek datetime (penting!)
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'])

# 2. Hapus data yang hilang (jika ada)
df_merged.dropna(inplace=True)

# 3. Urutkan data berdasarkan user dan waktu
# Ini penting untuk simulasi anomali sekuensial nanti
df_merged.sort_values(by=['user_id', 'timestamp'], inplace=True)

# 4. Buat dataset "Normal"
# Kita anggap semua data di Olist adalah normal
df_normal = df_merged.copy()
df_normal['Class'] = 0 # Beri label 0 (Normal)

print("Data Normal siap. Contoh:")
print(df_normal.head())

Data Normal siap. Contoh:
                               order_id                           user_id  \
71587  5f79b5b0931d63f1a42989eb65b9da6e  00012a2ce6f8dcda20d059ce98491703   
10466  a44895d095d7e0702b6a162fa2dbeced  000161a058600d5901f007fab4c27140   
68795  316a104623542e4d75189bb372bc5f8d  0001fd6190edaaf884bcaf3d49edf079   
45159  5825ce2e88d5346438686b0bba99e5ee  0002414f95344307404f0ace7a26f1d5   
6119   0ab7fb08086d4af9141453c91878ed7a  000379cdec625522490c315e70c7a9fb   

                timestamp  amount  Class  
71587 2017-11-14 16:08:26  114.74      0  
10466 2017-07-16 09:40:32   67.41      0  
68795 2017-02-28 11:06:43  195.42      0  
45159 2017-08-16 13:09:20  179.35      0  
6119  2018-04-02 13:42:17  107.01      0  


In [7]:
print("Memulai injeksi anomali...")
np.random.seed(42) # Agar hasilnya selalu sama (reproducible)

# --- Tipe 1: Anomali Nilai (Amount sangat tinggi) ---
# Ambil 200 sampel acak dari data normal untuk dijadikan fraud
df_anom_nilai = df_normal.sample(n=200)
# Kalikan nilainya 100x (atau 1000x)
df_anom_nilai['amount'] = df_anom_nilai['amount'] * 100
df_anom_nilai['Class'] = 1
print(f"1. Anomali Nilai dibuat: {len(df_anom_nilai)} transaksi")


# --- Tipe 2: Anomali Waktu (Transaksi jam 2-4 pagi) ---
df_anom_waktu = df_normal.sample(n=200)
# Ubah jam transaksi menjadi jam 3 pagi (plus menit acak)
df_anom_waktu['timestamp'] = df_anom_waktu['timestamp'].apply(
    lambda x: x.replace(hour=3, minute=np.random.randint(0, 59))
)
df_anom_waktu['Class'] = 1
print(f"2. Anomali Waktu dibuat: {len(df_anom_waktu)} transaksi")


# --- Tipe 3: Anomali Sekuensial (Paling penting utk LSTM) ---
# Ambil 50 user acak, lalu buat 5 transaksi beruntun (spam)
df_anom_sekuens_list = []
user_list = df_normal['user_id'].sample(n=50).unique()

for user in user_list:
    # Ambil 1 transaksi normal terakhir dari user ini
    base_transaction = df_normal[df_normal['user_id'] == user].iloc[-1:]
    
    if base_transaction.empty:
        continue # Lewati jika user tidak punya transaksi
        
    for i in range(5): # Buat 5 transaksi spam
        new_transaction = base_transaction.copy()
        
        # Mundurkan waktunya sedikit (misal 10-60 detik dari sebelumnya)
        new_timestamp = base_transaction['timestamp'].iloc[0] + pd.Timedelta(seconds=(i+1)*np.random.randint(10, 60))
        
        new_transaction['timestamp'] = new_timestamp
        new_transaction['amount'] = base_transaction['amount'].iloc[0] * np.random.uniform(0.8, 1.2) # Buat amount mirip2
        new_transaction['Class'] = 1
        
        df_anom_sekuens_list.append(new_transaction)

df_anom_sekuens = pd.concat(df_anom_sekuens_list)
print(f"3. Anomali Sekuensial dibuat: {len(df_anom_sekuens)} transaksi")

# Gabungkan semua data fraud
df_fraud_total = pd.concat([df_anom_nilai, df_anom_waktu, df_anom_sekuens])

Memulai injeksi anomali...
1. Anomali Nilai dibuat: 200 transaksi
2. Anomali Waktu dibuat: 200 transaksi
3. Anomali Sekuensial dibuat: 250 transaksi


In [8]:
# Gabungkan data normal asli dengan data fraud buatan
df_simulasi_final = pd.concat([df_normal, df_fraud_total])

# Acak urutan baris (PENTING!)
df_simulasi_final = df_simulasi_final.sample(frac=1).reset_index(drop=True)

# Cek hasil akhir
print("\n--- Hasil Akhir Dataset Simulasi ---")
print(df_simulasi_final['Class'].value_counts())

# Simpan ke folder Output
final_path = os.path.join(OUTPUT_PATH, 'simulasi_data_final.csv')
df_simulasi_final.to_csv(final_path, index=False)

print(f"\nDataset simulasi final berhasil disimpan di:\n{final_path}")
print("\nNotebook 1 Selesai!")


--- Hasil Akhir Dataset Simulasi ---
Class
0    103886
1       650
Name: count, dtype: int64

Dataset simulasi final berhasil disimpan di:
C:\FRAUD-Sinta2\Output\simulasi_data_final.csv

Notebook 1 Selesai!
